# Matter — commission bridge, baca, dan kendalikan

Bridge rumah pintar (`MatterDevice` + `MatterBridgeSimulator`) dan controller (`MatterController`) berbagi jaringan dalam memori. Dengan perangkat sungguhan pakai `UseUdp()` dan alamat perangkat di UDP 5540; controller tetap hanya-baca sampai `AllowWrites()`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Protocols.Matter, 0.30.0-preview.1"

## Kode onboarding dan TLV
Payload QR dan kode manual membawa discriminator dan passcode; pesan Matter membawa TLV.

In [ ]:
using System.Net;
using IoTCom.Net;
using IoTCom.Net.Protocols.Matter;
using IoTCom.Net.Transports;

var payload = MatterSetupPayload.FromQrCode("MT:Y.K9042C00KA0648G00");
Console.WriteLine($"{payload.Describe()} -> manual code {payload.ToManualCode()}");
var tlv = Tlv.Encode(TlvElement.Structure(TlvTag.Anonymous, TlvElement.UInt(TlvTag.Context(0), 42), TlvElement.String(TlvTag.Context(1), "Lampu")));
Console.WriteLine($"{Convert.ToHexString(tlv)} = {Tlv.Decode(tlv)}");

## Commission
PASE dengan passcode, fail-safe, atestasi, CSR, root dan NOC, lalu CASE — setiap langkah dilaporkan.

In [ ]:
var net = new InMemoryDatagramNetwork();
var at = new IPEndPoint(IPAddress.Parse("10.0.0.70"), 5540);
var bridge = MatterDevice.Create(o => { o.UseInMemory(net, at); o.ProductName = "IoTCom smart home bridge"; });
var house = new MatterBridgeSimulator(bridge);
await bridge.StartAsync();
var controller = MatterController.Create(o => o.UseInMemory(net).AllowWrites().TrustTestDevices());
var node = await controller.CommissionAsync(at, bridge.SetupPayload, 0x42, new Progress<MatterCommissioningStep>(s => Console.WriteLine($"  {s}")));
Console.WriteLine($"node 0x{node.NodeId:X} on fabric 0x{controller.FabricId:X} ({controller.CompressedFabricId})");

## Baca, invoke, subscribe
Pembacaan wildcard atas endpoint yang dijembatani, sebuah subscription, dan perintah On.

In [ ]:
foreach (var r in await node.ReadAsync([new AttributePath(null, MatterIds.BridgedDeviceBasicInformation, 5)]))
    Console.WriteLine($"endpoint {r.Path.Endpoint}: {r.Value!.AsString()}");
await node.SubscribeAsync([new AttributePath(null, MatterIds.OnOff, 0)], 0, 60, r => Console.WriteLine($"report {r.Path} = {r.Value}"));
await node.InvokeAsync(4, MatterIds.OnOff, 1);
await Task.Delay(300);
Console.WriteLine($"terrace light on: {house.Lights[2].OnOff.On}");
var readOnly = MatterController.Create(o => o.UseInMemory(net));
try { await readOnly.CommissionAsync(at, bridge.SetupPayload, 9); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
await readOnly.DisposeAsync();
await controller.DisposeAsync();
await bridge.DisposeAsync();

## Lebih lanjut
`iotcom matter code`, `iotcom matter device --bridge`, `iotcom matter commission <host> <code> --allow-write`, `iotcom matter demo --allow-write`, *Rumah pintar lewat Matter* di Gallery, dan sampel MatterHome. Lihat `docs/id/protocols/matter.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*